# 🎵 הורדת שירים ← קישור להורדה (Colab)

נותנים רשימת שמות שירים ← Colab מוריד את השמע (MP3), דוחס ל-ZIP, ומעלה לשירות שמחזיר **קישור הורדה**. הכל רץ בענן — שום דבר לא נשמר במחשב שלך.

**איך מריצים:** לחץ על כפתור ה⁩▶⁩ של כל תא לפי הסדר (או תפריט Runtime ← Run all). בתא השני — הדבק את השירים שלך.

### 1️⃣ התקנת הכלים (רץ פעם אחת לכל סשן)

In [ ]:
# yt-dlp (הורדה), ffmpeg (המרה ל-mp3), ffsend (העלאה ל-Send)
!pip -q install --upgrade yt-dlp
!apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://github.com/timvisee/ffsend/releases/download/v0.2.76/ffsend-v0.2.76-linux-x64-static -O /usr/local/bin/ffsend && chmod +x /usr/local/bin/ffsend
print('✅ הכל מוכן')

### 2️⃣ רשימת השירים שלך
שיר אחד בכל שורה. הכי טוב בפורמט `שם אמן - שם שיר`. אפשר להוסיף `official audio` לשיר שיורד בגרסה לא נכונה.

In [ ]:
songs = """
Idan Raichel - Mimaamakim
Hadag Nachash - Zman Lehitorer
Rita - Shvil Habricha
"""

### 3️⃣ הורדה + דחיסה ל-ZIP

In [ ]:
import subprocess, zipfile, pathlib, os

OUT = pathlib.Path('downloads'); OUT.mkdir(exist_ok=True)
song_list = [s.strip() for s in songs.strip().splitlines() if s.strip() and not s.strip().startswith('#')]
print(f'{len(song_list)} שירים להורדה\n')

ok, fail = [], []
for i, song in enumerate(song_list, 1):
    print(f'[{i}/{len(song_list)}] {song}')
    r = subprocess.run([
        'yt-dlp', f'ytsearch1:{song}', '--no-playlist',
        '-x', '--audio-format', 'mp3', '--audio-quality', '0',
        '--add-metadata', '--embed-thumbnail',
        '-o', str(OUT / '%(title)s [%(id)s].%(ext)s'),
        '--print', 'after_move:filepath', '--no-warnings', '--quiet',
    ], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        print('   ✅', pathlib.Path(r.stdout.strip().splitlines()[-1]).name)
        ok.append(song)
    else:
        last = (r.stderr.strip().splitlines() or [''])[-1]
        print('   ❌ נכשל:', last[:180])
        fail.append(song)

print(f'\nסיכום: {len(ok)} הצליחו, {len(fail)} נכשלו')

ZIP = 'songs.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob('*.mp3'):
        z.write(f, f.name)
print(f'\n📦 {ZIP} — {os.path.getsize(ZIP)/1024/1024:.1f} MB')

### 4️⃣ העלאה ← קישור הורדה
מעלה ל-`send.magicode.me`. אם זה לא עובד מסיבה כלשהי — נופל אוטומטית ל-`0x0.st`.

In [ ]:
import subprocess, re

SEND_HOST = 'https://send.magicode.me/'   # אפשר להחליף ל-Send אחר

def find_url(t):
    m = re.search(r'https?://\S+', t or '')
    return m.group(0).rstrip('.,) ') if m else None

link = None
print(f'מעלה ל-{SEND_HOST} ...')
r = subprocess.run(['ffsend', 'upload', '--host', SEND_HOST, 'songs.zip'], capture_output=True, text=True)
link = find_url(r.stdout) or find_url(r.stderr)
if not link:
    print('Send לא עבד, נופל ל-0x0.st ...')
    if r.stderr.strip():
        print('  ', r.stderr.strip().splitlines()[-1][:180])
    r = subprocess.run(['curl', '-s', '-F', 'file=@songs.zip', 'https://0x0.st'], capture_output=True, text=True)
    link = find_url(r.stdout)

print('\n' + '=' * 50)
if link:
    print('🔗 קישור ההורדה שלך:\n\n   ' + link + '\n')
else:
    print('❌ לא הצלחתי ליצור קישור. נסה שוב או שנה את SEND_HOST.')

---
**שיר נכשל / גרסה לא נכונה?** ערוך את השורה בתא 2 (למשל הוסף `official audio` או שם האלבום) והרץ שוב מתא 3.

**הונדרט שירים?** Colab מחזיק מעם שעות. פשוט תן לו לרוץ — הקישור מודפס בסוף.

**YouTube מבקש אימות ("Sign in to confirm...")?** לפעמים קורה בשרתי ענן. נסה שוב מאוחר יותר, או פנה אלי9 לעזרה עם cookies.